# Class 1 Combined Notebook

This notebook combines `random_walk.py` and `meeting_probability.py` from Class 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N_STEPS = 200
N_WALKERS = 1000
rng = np.random.default_rng(42)

steps = rng.choice([-1, 1], size=(N_WALKERS, N_STEPS))
positions = np.hstack([np.zeros((N_WALKERS, 1)), np.cumsum(steps, axis=1)])
step_axis = np.arange(N_STEPS + 1)

mean_x = positions.mean(axis=0)
mean_x2 = (positions**2).mean(axis=0)

# Step increments for P(Δx)
delta_x = steps.flatten()  # all individual coin-flip steps: ±1
dx_vals = np.array([-1, 1])
P_dx_theory = np.array([0.5, 0.5])  # each equally probable

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(8, 13))

fig.patch.set_facecolor("#1a1d27")

# Plot 1: <x> vs step
ax1.plot(step_axis, mean_x, color="red", lw=1.5)
ax1.axhline(0, color="white", lw=1, ls="--")
ax1.set_ylabel(r"$\langle x \rangle$")
ax1.set_title(r"$\langle x \rangle$ vs Step Number")

# Plot 2: <x^2> vs step
ax2.plot(step_axis, mean_x2, color="red", lw=1.5, label="Simulated")
ax2.plot(step_axis, step_axis, color="white", lw=1, ls="--", label=r"Theory: $N$")
ax2.set_ylabel(r"$\langle x^2 \rangle$")
ax2.set_title(r"$\langle x^2 \rangle$ vs Step Number")
ax2.legend()

# Plot 3: P(Δx) vs Δx
counts = np.array([(delta_x == v).sum() for v in dx_vals], dtype=float)
counts /= counts.sum()  # normalize to probability

ax3.bar(dx_vals, counts, width=0.4, color="red", alpha=0.7, label=r"Simulated $P(\Delta x)$")
ax3.vlines(dx_vals, 0, P_dx_theory, color="white", lw=2, ls="--", label=r"Theory: $P(\Delta x)=0.5$")
ax3.scatter(dx_vals, P_dx_theory, color="white", zorder=5, s=60)
ax3.set_xticks([-1, 1])
ax3.set_xticklabels([r"$\Delta x = -1$", r"$\Delta x = +1$"])
ax3.set_xlim(-2, 2)
ax3.set_ylim(0, 0.7)
ax3.set_ylabel(r"$P(\Delta x)$")
ax3.set_title(r"Step Distribution  $P(\Delta x)$ vs $\Delta x$  (coin flip)")
ax3.legend()

# Plot 4: Multiple individual random walks (coin flip)
N_SHOW = 20  # number of individual walks to display
for i in range(N_SHOW):
    ax4.plot(step_axis, positions[i], color="red", lw=0.8, alpha=0.4)
ax4.plot(step_axis, mean_x, color="white", lw=1.8, ls="--", label="⟨x⟩ (ensemble mean)")
ax4.axhline(0, color="gray", lw=0.6, ls=":")
ax4.set_xlabel("Step Number")
ax4.set_ylabel("x")
ax4.set_title(f"Multiple Random Walks from Coin Flip  ({N_SHOW} walkers)")
ax4.legend()

plt.tight_layout()
plt.savefig("random_walk_statistics.png", dpi=150)
print("Saved.")
plt.show()

## Two Random Walkers: Meeting Probability

In [ ]:
"""
Two Random Walkers - Probability of Meeting after N steps

Two walkers start at origin, take simultaneous +/-1 steps.
Relative displacement d = x1 - x2 is itself a random walk.
They meet when d = 0.

Exact result:   P(N) = C(2N, N) / 4^N
Asymptotic:     P(N) ~ 1 / sqrt(pi * N)
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gammaln

N_MAX = 300      # max steps
N_TRIALS = 50000 # number of independent pairs per step
rng = np.random.default_rng(42)

# Simulation
steps1 = rng.choice([-1, 1], size=(N_TRIALS, N_MAX))
steps2 = rng.choice([-1, 1], size=(N_TRIALS, N_MAX))

x1 = np.cumsum(steps1, axis=1)
x2 = np.cumsum(steps2, axis=1)

# P_sim[n] = fraction of trials where walkers meet at step n+1
P_sim = (x1 == x2).mean(axis=0)

step_axis = np.arange(1, N_MAX + 1)

# Theory
log_P_exact = gammaln(2 * step_axis + 1) - 2 * gammaln(step_axis + 1) - step_axis * np.log(4)
P_exact = np.exp(log_P_exact)

# Asymptotic: 1 / sqrt(pi * N)
P_asymptotic = 1.0 / np.sqrt(np.pi * step_axis)

# Print table of key values
key_steps = [1, 2, 5, 10, 20, 50, 100, 200, 300]
print(f"{'N':>5}  {'P_exact':>12}  {'P_asymptotic':>14}  {'P_sim':>10}")
print("-" * 48)
for n in key_steps:
    idx = n - 1
    print(f"{n:>5}  {P_exact[idx]:>12.6f}  {P_asymptotic[idx]:>14.6f}  {P_sim[idx]:>10.6f}")

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8))

# Panel 1: P(N) vs N (linear scale)
ax1.plot(step_axis, P_sim, color="red", lw=1.2, alpha=0.8, label="Simulation")
ax1.plot(step_axis, P_exact, color="white", lw=1.8, ls="--", label=r"Exact: $\binom{2N}{N}/4^N$")
ax1.plot(step_axis, P_asymptotic, color="gray", lw=1.2, ls=":", label=r"Asymptotic: $1/\sqrt{\pi N}$")
ax1.set_xlabel("Step number N")
ax1.set_ylabel("P( meet at step N )")
ax1.set_title("Probability of Two Walkers Meeting after N Steps")
ax1.legend()
ax1.set_xlim(0, N_MAX)
ax1.set_ylim(bottom=0)

# Annotate key probability values on the graph
for n in [1, 5, 10, 50, 100, 200]:
    idx = n - 1
    p = P_exact[idx]
    ax1.annotate(
        f"N={n}\nP={p:.4f}",
        xy=(n, p),
        xytext=(n + 8, p + 0.02),
        fontsize=7,
        color="white",
        arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
    )

# Panel 2: log-log scale to see power law
ax2.loglog(step_axis, P_sim, color="red", lw=1.2, alpha=0.8, label="Simulation")
ax2.loglog(step_axis, P_exact, color="white", lw=1.8, ls="--", label=r"Exact: $\binom{2N}{N}/4^N$")
ax2.loglog(step_axis, P_asymptotic, color="gray", lw=1.2, ls=":", label=r"Slope $= -\frac{1}{2}$  $(1/\sqrt{\pi N})$")
ax2.set_xlabel("Step number N  (log scale)")
ax2.set_ylabel("P (log scale)")
ax2.set_title(r"Log-Log Plot  -  confirms $P(N) \sim N^{-1/2}$")
ax2.legend()

plt.tight_layout()
plt.savefig("meeting_probability.png", dpi=150)
print("Saved.")
plt.show()